# Module 02 - Tensors and Matmul

Use this notebook as the working space for the Module 02 exercises. Keep `notebooks/clean/` pristine; work in the copy created under `notebooks/solutions/`.

In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from g2c.tensors import (
    TinyArray,
    broadcast_shapes,
    linear,
    matmul_loops,
    matmul_numpy,
    matmul_torch,
    softmax,
)

np.random.seed(0)
torch.manual_seed(0)

mps_available = torch.backends.mps.is_available()
mps_available

## Exercise 1 - Shape tracing

For each expression, write the output shape or mark it invalid. For invalid expressions, name the mismatched dimensions.

In [ ]:
"Question: A @ B where A=(2, 3), B=(3, 4)"
"Answer: "

"Question: B @ A for the same A and B"
"Answer: "

"Question: x @ W + b where x=(5, 2), W=(2, 6), b=(6,)"
"Answer: "

"Question: x @ W + b_bad where b_bad=(5,)"
"Answer: "

## Exercise 2 - Manual matmul

Compute the product by hand first. After your matmul implementations pass their tests, verify the same result with all three implementations.

In [ ]:
A = [[1.0, 2.0, 0.0], [-1.0, 3.0, 4.0]]
B = [[2.0, 1.0], [0.0, -2.0], [5.0, 3.0]]

manual_product = [
    # TODO: fill in the two output rows after computing by hand.
]

# Uncomment after filling manual_product and implementing matmul.
# np.testing.assert_allclose(matmul_loops(A, B), manual_product)
# np.testing.assert_allclose(matmul_numpy(np.array(A), np.array(B)), manual_product)
# np.testing.assert_allclose(matmul_torch(torch.tensor(A), torch.tensor(B)), torch.tensor(manual_product))

manual_product

## Exercise 3 - Benchmark and interpret matmul

Before running, write down your predicted ordering: Python loops, NumPy, PyTorch CPU, PyTorch MPS. Then benchmark and explain the result.

In [ ]:
predicted_order_fastest_to_slowest = [
    # TODO: fill before running the benchmark.
]

predicted_order_fastest_to_slowest

In [ ]:
def median_seconds(fn, repeats: int = 5) -> float:
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = fn()
        if isinstance(result, torch.Tensor) and result.device.type == "mps":
            torch.mps.synchronize()
        times.append(time.perf_counter() - start)
    return float(np.median(times))


sizes = [32, 64, 128, 256]
results = []

for n in sizes:
    A_np = np.random.randn(n, n).astype(np.float32)
    B_np = np.random.randn(n, n).astype(np.float32)
    A_t = torch.tensor(A_np)
    B_t = torch.tensor(B_np)

    if n <= 128:
        results.append({"size": n, "engine": "loops", "seconds": median_seconds(lambda: matmul_loops(A_np.tolist(), B_np.tolist()), repeats=1)})

    results.append({"size": n, "engine": "numpy", "seconds": median_seconds(lambda: matmul_numpy(A_np, B_np))})
    results.append({"size": n, "engine": "torch_cpu", "seconds": median_seconds(lambda: matmul_torch(A_t, B_t))})

    if mps_available:
        A_mps = A_t.to("mps")
        B_mps = B_t.to("mps")
        matmul_torch(A_mps, B_mps)
        torch.mps.synchronize()
        results.append({"size": n, "engine": "torch_mps", "seconds": median_seconds(lambda: matmul_torch(A_mps, B_mps))})

results[:5]

In [ ]:
for engine in sorted({row["engine"] for row in results}):
    rows = [row for row in results if row["engine"] == engine]
    plt.plot([row["size"] for row in rows], [row["seconds"] for row in rows], marker="o", label=engine)

plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("matrix size n for n x n @ n x n")
plt.ylabel("median seconds")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

### Benchmark notes

- Best constant factor:
- Size where MPS starts to win:
- Why Python loops become unusable:

## Exercise 4 - Broadcasting predictions

Predict each result shape before running the verification cell. For valid cases, note which dimensions stretch.

In [ ]:
broadcast_predictions = [
    ((3, 4), (4,), "TODO"),
    ((2, 1, 3), (1, 5, 1), "TODO"),
    ((5, 1), (1, 4), "TODO"),
    ((2, 3), (3, 2), "TODO"),
    ((), (2, 3, 4), "TODO"),
]

for shape_a, shape_b, predicted in broadcast_predictions:
    try:
        actual = broadcast_shapes(shape_a, shape_b)
    except ValueError:
        actual = "invalid"
    print(f"{shape_a} with {shape_b}: predicted={predicted}, actual={actual}")

In [ ]:
# Add at least two TinyArray examples that verify your broadcasting predictions.

example_a = TinyArray([1, 2, 3, 4, 5, 6], (2, 3))
example_b = TinyArray([10, 20, 30], (3,))

# Uncomment after TinyArray broadcasting is implemented.
# (example_a + example_b).to_nested()

## Exercise 5 - Softmax stability

Explain why naive softmax over `[1000.0, 1001.0]` overflows in float32. Then verify that the stable softmax is invariant to adding the same constant to every logit.

In [ ]:
logits = torch.tensor([1000.0, 1001.0])

stable = softmax(logits, dim=0)
shifted = softmax(logits + 12345.0, dim=0)

print(stable)
print(shifted)
print(torch.allclose(stable, shifted, atol=1e-6))

### Softmax notes

- Why the naive version overflows:
- Why subtracting the max preserves the probabilities:

## Exercise 6 - Tiny classifier forward pass

Build a one-layer classifier with random weights. Annotate every shape and check that each probability row sums to 1.

In [ ]:
batch = 4
in_features = 3
num_classes = 5

x = torch.randn(batch, in_features)
W = torch.randn(in_features, num_classes)
b = torch.zeros(num_classes)

logits = linear(x, W, b)
probs = softmax(logits, dim=-1)

shape_report = {
    "x": tuple(x.shape),
    "W": tuple(W.shape),
    "b": tuple(b.shape),
    "logits": tuple(logits.shape),
    "probs": tuple(probs.shape),
    "row_sums": probs.sum(dim=-1),
}

assert logits.shape == (batch, num_classes)
assert probs.shape == (batch, num_classes)
assert torch.allclose(probs.sum(dim=-1), torch.ones(batch), atol=1e-6)

shape_report